In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder

from src.data import load_deeploc, split_by_partition, encode_many
from src.models import ProteinCNN
from src.train import set_seed, train_cnn, predict

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
df = load_deeploc("../data/deeploc_data.fasta")   # change the file name if yours differs
print(df.shape)
display(df.head())
print(df.part.value_counts(dropna=False))
print(df.location.value_counts())
print(df.membrane.value_counts())
print(df.seq.str.len().describe())

(14004, 5)


,id,location,membrane,part,seq
0,Q9H400,Cell.membrane,M,test,MGLPVSWAPPALWVLGCCALLLSLWALCTACRRPEDAVAPRKRARR...
1,Q5I0E9,Cell.membrane,M,NaN,MEVLEEPAPGPGGADAAERRGLRRLLLSGFQEELRALLVLAGPAFL...
2,P63033,Cell.membrane,M,NaN,MMKTLSSGNCTLNVPAKNSYRMVVLGASRVGKSSIVSRFLNGRFED...
3,Q9NR71,Cell.membrane,M,NaN,MAKRTFSNLETFLIFLLVMMSAITVALLSLLFITSGTIENHKDLGG...
4,Q86XT9,Cell.membrane,M,NaN,MGNCQAGHNLHLCLAHHPPLVCATLILLLLGLSGLGLGSFLLTHRT...


part
NaN     11231
test     2773
Name: count, dtype: int64
location
Nucleus                  4043
Cytoplasm                2542
Extracellular            1973
Mitochondrion            1510
Cell.membrane            1340
Endoplasmic.reticulum     862
Plastid                   757
Golgi.apparatus           356
Lysosome/Vacuole          321
Peroxisome                154
Cytoplasm-Nucleus         146
Name: count, dtype: int64
membrane
U    5342
S    5109
M    3553
Name: count, dtype: int64
count    14004.00000
mean       522.44166
std        448.06671
min         40.00000
25%        249.00000
50%        421.00000
75%        649.00000
max      13100.00000
Name: seq, dtype: float64


In [3]:
from src.data import (shared_kmer_matrix, cluster_from_shared,
                      split_train_val_by_cluster)

keep = df.location != "Cytoplasm-Nucleus"
test = df[keep & (df.part == "test")].reset_index(drop=True)
trainval = df[keep & df.part.isna()].reset_index(drop=True)

shared, n_kmers = shared_kmer_matrix(trainval.seq.tolist())

for min_shared in [10, 20, 40]:
    for min_frac in [0.05, 0.1, 0.2]:
        g = cluster_from_shared(shared, n_kmers, min_frac, min_shared)
        s = pd.Series(g).value_counts()
        print(f"min_shared={min_shared:3d} min_frac={min_frac:.2f} -> "
              f"clusters {len(s):5d}, largest {s.iloc[0]:5d} ({s.iloc[0] / len(g):.0%})")

min_shared= 10 min_frac=0.05 -> clusters  7496, largest   307 (3%)
min_shared= 10 min_frac=0.10 -> clusters  8285, largest   199 (2%)
min_shared= 10 min_frac=0.20 -> clusters  8960, largest   189 (2%)
min_shared= 20 min_frac=0.05 -> clusters  7866, largest   145 (1%)
min_shared= 20 min_frac=0.10 -> clusters  8402, largest   145 (1%)
min_shared= 20 min_frac=0.20 -> clusters  9021, largest   145 (1%)
min_shared= 40 min_frac=0.05 -> clusters  8527, largest    23 (0%)
min_shared= 40 min_frac=0.10 -> clusters  8739, largest    16 (0%)
min_shared= 40 min_frac=0.20 -> clusters  9223, largest    15 (0%)


In [4]:
MIN_SHARED = 10     
MIN_FRAC = 0.05     

groups = cluster_from_shared(shared, n_kmers, MIN_FRAC, MIN_SHARED)
sizes = pd.Series(groups).value_counts()
print("clusters:", len(sizes), "| largest cluster:", sizes.iloc[0], "of", len(trainval))
print("exact sequence overlap trainval/test:", len(set(trainval.seq) & set(test.seq)))

train, val = split_train_val_by_cluster(trainval, groups, seed=0)
assert set(groups[train.index]).isdisjoint(set(groups[val.index]))
print("train:", len(train), "| val:", len(val))
display(pd.concat([train.location.value_counts(), val.location.value_counts()], axis=1,
                  keys=["train", "val"]))

le = LabelEncoder().fit(train.location)
y_train = le.transform(train.location)
y_val = le.transform(val.location)
class_names = list(le.classes_)
print(class_names)

clusters: 7496 | largest cluster: 307 of 11085
exact sequence overlap trainval/test: 0
train: 8867 | val: 2218


,train,val
location,,
Nucleus,2588,647
Cytoplasm,1628,406
Extracellular,1264,316
Mitochondrion,966,242
Cell.membrane,853,214
Endoplasmic.reticulum,551,138
Plastid,484,121
Golgi.apparatus,229,57
Lysosome/Vacuole,205,52


['Cell.membrane', 'Cytoplasm', 'Endoplasmic.reticulum', 'Extracellular', 'Golgi.apparatus', 'Lysosome/Vacuole', 'Mitochondrion', 'Nucleus', 'Peroxisome', 'Plastid']


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

vec = CountVectorizer(analyzer="char", ngram_range=(1, 3), lowercase=False)
X_tr_kmer = normalize(vec.fit_transform(train.seq), norm="l1").astype(np.float32)
X_val_kmer = normalize(vec.transform(val.seq), norm="l1").astype(np.float32)
print(X_tr_kmer.shape)

results = []
for seed in [0, 1, 2]:
    clf = XGBClassifier(
        n_estimators=400, learning_rate=0.1, max_depth=6,
        subsample=0.8, colsample_bytree=0.5,
        tree_method="hist", device="cuda",
        eval_metric="mlogloss", early_stopping_rounds=20,
        random_state=seed,
    )
    clf.fit(X_tr_kmer, y_train, eval_set=[(X_val_kmer, y_val)], verbose=False)
    pred = clf.predict(X_val_kmer)
    results.append({"model": "kmer+xgboost", "seed": seed,
                    "val_acc": accuracy_score(y_val, pred),
                    "val_macro_f1": f1_score(y_val, pred, average="macro")})
    print(results[-1])

(8867, 8584)


C:\dev\protein-env\Lib\site-packages\xgboost\core.py:569: UserWarning: [12:11:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


{'model': 'kmer+xgboost', 'seed': 0, 'val_acc': 0.5207394048692516, 'val_macro_f1': 0.3410394797725106}
{'model': 'kmer+xgboost', 'seed': 1, 'val_acc': 0.5229936880072137, 'val_macro_f1': 0.344382728108556}
{'model': 'kmer+xgboost', 'seed': 2, 'val_acc': 0.5225428313796213, 'val_macro_f1': 0.3466976884517488}


In [6]:
X_tr = encode_many(train.seq.tolist(), max_len=1000)
X_val = encode_many(val.seq.tolist(), max_len=1000)
print(X_tr.shape, X_val.shape)

histories = {}
for seed in [0, 1, 2]:
    print(f"===== seed {seed} =====")
    set_seed(seed)
    model = ProteinCNN(n_classes=len(class_names))
    model, hist = train_cnn(model, X_tr, y_train, X_val, y_val, device, seed)
    pred = predict(model, X_val, device)
    histories[seed] = hist
    results.append({"model": "cnn", "seed": seed,
                    "val_acc": accuracy_score(y_val, pred),
                    "val_macro_f1": f1_score(y_val, pred, average="macro")})
    print(results[-1])

(8867, 1000) (2218, 1000)
===== seed 0 =====
epoch  1  loss 1.9781  val acc 0.454  val macro-F1 0.248
epoch  2  loss 1.5287  val acc 0.532  val macro-F1 0.318
epoch  3  loss 1.3089  val acc 0.525  val macro-F1 0.299
epoch  4  loss 1.1687  val acc 0.589  val macro-F1 0.370
epoch  5  loss 1.0553  val acc 0.598  val macro-F1 0.412
epoch  6  loss 0.9634  val acc 0.606  val macro-F1 0.413
epoch  7  loss 0.8856  val acc 0.615  val macro-F1 0.413
epoch  8  loss 0.8152  val acc 0.608  val macro-F1 0.403
epoch  9  loss 0.7310  val acc 0.599  val macro-F1 0.398
epoch 10  loss 0.6907  val acc 0.593  val macro-F1 0.408
epoch 11  loss 0.6372  val acc 0.601  val macro-F1 0.398
early stopping
{'model': 'cnn', 'seed': 0, 'val_acc': 0.60595130748422, 'val_macro_f1': 0.41343354721414016}
===== seed 1 =====
epoch  1  loss 1.9578  val acc 0.475  val macro-F1 0.241
epoch  2  loss 1.4745  val acc 0.530  val macro-F1 0.276
epoch  3  loss 1.2693  val acc 0.571  val macro-F1 0.336
epoch  4  loss 1.1372  val ac

In [8]:
res = pd.DataFrame(results)
summary = res.groupby("model")[["val_acc", "val_macro_f1"]].agg(["mean", "std"]).round(3)
display(summary)
res.to_csv("../outputs/baseline_cnn_results.csv", index=False)

val_acc        val_macro_f1       
                mean    std         mean    std
model                                          
cnn            0.614  0.007        0.426  0.012
kmer+xgboost   0.522  0.001        0.344  0.003

In [9]:
from sklearn.metrics import classification_report

pred = predict(model, X_val, device)
print(classification_report(y_val, pred, target_names=class_names, digits=3))

                       precision    recall  f1-score   support

        Cell.membrane      0.609     0.664     0.635       214
            Cytoplasm      0.497     0.569     0.530       406
Endoplasmic.reticulum      0.379     0.478     0.423       138
        Extracellular      0.828     0.687     0.751       316
      Golgi.apparatus      0.214     0.105     0.141        57
     Lysosome/Vacuole      0.000     0.000     0.000        52
        Mitochondrion      0.549     0.653     0.596       242
              Nucleus      0.726     0.784     0.754       647
           Peroxisome      0.000     0.000     0.000        25
              Plastid      0.746     0.413     0.532       121

             accuracy                          0.621      2218
            macro avg      0.455     0.435     0.436      2218
         weighted avg      0.609     0.621     0.609      2218



C:\dev\protein-env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\dev\protein-env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\dev\protein-env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
